# MLP Backpropagation from Scratch

In [1]:
import numpy as np
import pandas as pd

## 1. Regression Problem

In [2]:
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 6]], columns=['cgpa', 'profile_score', 'lpa'])

In [3]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,6


### a. Parameter initialization

Model Architecture
- 2 inputs, 2 hidden dim, 1 output
- Activation linear
- Loss: MSE

In [4]:
def initialize_parameters(layer_dims):

    np.random.seed(3)
    parameters = {}

    L = len(layer_dims)

    for l in range(1, L):

        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

In [5]:
initialize_parameters([2, 2, 1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

### b. Forward propagation

In [6]:
def linear_forward(A_prev, W, b):
    Z = (W.T @ A_prev) + b
    return Z

In [7]:
# Forward prop
def L_layer_forward(X, parameters):

    A = X
    L = len(parameters) // 2

    for l in range(1, L+1):
        A_prev = A
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]

        # print("A" + str(l-1) + ": ", A_prev)
        # print("W" + str(l) + ": ", Wl)
        # print("b" + str(l) + ": ", bl)
        # print("--"*20)

        A = linear_forward(A_prev, Wl, bl)
        # print("A"+str(l)+": ", A)
        # print("**"*20)

    return A, A_prev

In [8]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2, 1) # Shape(no. of features, no. of training examples)
y = df[['lpa']].values[0][0]

# Parameter initialization
parameters = initialize_parameters([2, 2, 1])

In [9]:
y_hat, A1 = L_layer_forward(X, parameters)
y_hat = y_hat[0][0]

### c. Backward propagation

In [10]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):
    dZ2 = 2 * (y_hat - y) # Gradient of loss wrt y_hat

    # Save old W2 (needed for hidden gradients)
    w20 = parameters['W2'][0][0]
    w21 = parameters['W2'][1][0]

    # ----- hidden layer error -----
    dZ1_0 = w20 * dZ2
    dZ1_1 = w21 * dZ2

    # ----- gradients W1 -----
    parameters['W1'][0][0] -= lr * (X[0][0] * dZ1_0)
    parameters['W1'][1][0] -= lr * (X[1][0] * dZ1_0)

    parameters['W1'][0][1] -= lr * (X[0][0] * dZ1_1)
    parameters['W1'][1][1] -= lr * (X[1][0] * dZ1_1)

    # ----- gradients b1 -----
    parameters['b1'][0][0] -= lr * dZ1_0
    parameters['b1'][1][0] -= lr * dZ1_1

    # ----- gradients W2 -----
    parameters['W2'][0][0] -= lr * (A1[0][0] * dZ2)
    parameters['W2'][1][0] -= lr * (A1[1][0] * dZ2)

    # ----- gradients b2 -----
    parameters['b2'][0][0] -= lr * dZ2


In [11]:
update_parameters(parameters, y, y_hat, A1, X)

In [12]:
parameters

{'W1': array([[0.105888, 0.105888],
        [0.105888, 0.105888]]),
 'b1': array([[0.000736],
        [0.000736]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.00736]])}

### d. Complete Epochs

In [13]:
# Epochs implementation

parameters = initialize_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):

    loss = []

    for j in range(df.shape[0]):

        X = df[['cgpa', 'profile_score']].values[j].reshape(2, 1)
        y = df[['lpa']].values[j][0]

        # Forward propagation
        y_hat, A1 = L_layer_forward(X, parameters)
        y_hat = y_hat[0][0]

        # Backward propagation
        update_parameters(parameters, y, y_hat, A1, X)

        loss.append((y - y_hat)**2)

    print("Epoch", i+1, 'Loss: ', np.array(loss).mean())

Epoch 1 Loss:  23.428509075497914
Epoch 2 Loss:  17.765273806116372
Epoch 3 Loss:  10.05282871164664
Epoch 4 Loss:  3.6939321270652505
Epoch 5 Loss:  1.062305048448049


In [14]:
parameters

{'W1': array([[0.26750282, 0.26750282],
        [0.37150995, 0.37150995]]),
 'b1': array([[0.02694161],
        [0.02694161]]),
 'W2': array([[0.44855407],
        [0.44855407]]),
 'b2': array([[0.11813432]])}

### e. Keras baseline

In [15]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Input, Dense

2026-02-09 11:21:50.623140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770636110.825183      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770636110.884654      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770636111.358211      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770636111.358255      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770636111.358258      55 computation_placer.cc:177] computation placer alr

In [16]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2, activation='linear'),
    Dense(1, activation='linear')
])

2026-02-09 11:22:06.302138: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
new_weights = [np.array([[0.1, 0.1], [0.1, 0.1]], dtype=np.float32), np.array([0, 0], dtype=np.float32), np.array([[0.1], [0.1]], dtype=np.float32), np.array([0], dtype=np.float32)]

In [19]:
model.get_weights()

[array([[ 0.22323465,  0.93319666],
        [-0.1541481 ,  0.54005873]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[ 0.8597647],
        [-1.0169684]], dtype=float32),
 array([0.], dtype=float32)]

In [20]:
model.set_weights(new_weights)

In [21]:
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [22]:
optimizer = keras.optimizers.SGD(learning_rate=0.001)
model.compile(loss='mean_squared_error', optimizer=optimizer)

In [23]:
model.fit(df.iloc[:,0:-1].values, df['lpa'].values, epochs=5, verbose=1, batch_size=1)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 27.6428  
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 16.6471 
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 12.5789 
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.6855 
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.6949 


In [24]:
model.get_weights()

[array([[0.25951928, 0.25951928],
        [0.3652592 , 0.3652592 ]], dtype=float32),
 array([0.02602001, 0.02602001], dtype=float32),
 array([[0.4385959],
        [0.4385959]], dtype=float32),
 array([0.11588168], dtype=float32)]

## 2. Classification Problem

In [25]:
df = pd.DataFrame([[8, 8, 1], [7, 9, 1], [6, 10, 0], [5, 5, 0]], columns=['cgpa', 'profile_score', 'placed'])

In [26]:
df.head()

,cgpa,profile_score,placed
0,8,8,1
1,7,9,1
2,6,10,0
3,5,5,0


In [32]:
# Utility Functions
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def linear_forward(A_prev, W, b):
    Z = (W.T @ A_prev) + b
    return sigmoid(Z)

### a. Parameter initialization

Model Architecture
- 2 inputs, 2 hidden dim, 1 output
- Activation: Sigmoid (All neurons)
- Loss: BCE

### b. Forward Propagation

- Same as regression

In [28]:
# Forward prop
def L_layer_forward(X, parameters):

    A = X
    L = len(parameters) // 2

    for l in range(1, L+1):
        A_prev = A
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]

        # print("A" + str(l-1) + ": ", A_prev)
        # print("W" + str(l) + ": ", Wl)
        # print("b" + str(l) + ": ", bl)
        # print("--"*20)

        A = linear_forward(A_prev, Wl, bl)
        # print("A"+str(l)+": ", A)
        # print("**"*20)

    return A, A_prev

### c. Backward propagation

In [29]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):
    # Save old W2 (needed for hidden gradients)
    w20 = parameters['W2'][0][0]
    w21 = parameters['W2'][1][0]

    # Activation from previous layer to output
    o1 = A1[0][0]
    o2 = A1[1][0]

    # Input data (features)
    x1 = X[0][0]
    x2 = X[1][0]

    # dL/dz_final = dL/dy_pred * dy_pred/dz_final
    dl_dzf = lr * (y - y_hat) # negative sign cancelled out for updates

    # ----- gradients W1 -----
    parameters['W1'][0][0] += dl_dzf * w20 * o1 * (1-o1) * x1
    parameters['W1'][1][0] += dl_dzf * w20 * o1 * (1-o1) * x2

    parameters['W1'][0][1] += dl_dzf * w21 * o2 * (1-o2) * x1
    parameters['W1'][1][1] += dl_dzf * w21 * o2 * (1-o2) * x2

    # ----- gradients b1 -----
    parameters['b1'][0][0] += dl_dzf * w20 * o1 * (1-o1)
    parameters['b1'][1][0] += dl_dzf * w21 * o2 * (1-o2)

    # ----- gradients W2 -----
    parameters['W2'][0][0] += dl_dzf * o1
    parameters['W2'][1][0] += dl_dzf * o2
    
    # ----- gradients b2 -----
    parameters['b2'][0][0] += dl_dzf

### d. Complete Epochs

In [33]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 50

for i in range(epochs):

    Loss = []

    for j in range(df.shape[0]):

        X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
        y = df[['placed']].values[j][0]

        # Parameter initialization
        y_hat, A1 = L_layer_forward(X,parameters)
        y_hat = y_hat[0][0]

        update_parameters(parameters,y,y_hat,A1,X)

        Loss.append(-y*np.log(y_hat) - (1-y)*np.log(1-y_hat))

    print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

Epoch -  1 Loss -  0.6941717896838971
Epoch -  2 Loss -  0.6941618440088202
Epoch -  3 Loss -  0.6941519427402767
Epoch -  4 Loss -  0.6941420856639815
Epoch -  5 Loss -  0.6941322725667136
Epoch -  6 Loss -  0.6941225032363104
Epoch -  7 Loss -  0.6941127774616618
Epoch -  8 Loss -  0.6941030950327047
Epoch -  9 Loss -  0.6940934557404181
Epoch -  10 Loss -  0.6940838593768177
Epoch -  11 Loss -  0.6940743057349501
Epoch -  12 Loss -  0.6940647946088873
Epoch -  13 Loss -  0.6940553257937219
Epoch -  14 Loss -  0.6940458990855618
Epoch -  15 Loss -  0.6940365142815246
Epoch -  16 Loss -  0.6940271711797323
Epoch -  17 Loss -  0.6940178695793071
Epoch -  18 Loss -  0.6940086092803646
Epoch -  19 Loss -  0.6939993900840101
Epoch -  20 Loss -  0.6939902117923326
Epoch -  21 Loss -  0.6939810742084004
Epoch -  22 Loss -  0.6939719771362554
Epoch -  23 Loss -  0.6939629203809086
Epoch -  24 Loss -  0.6939539037483344
Epoch -  25 Loss -  0.6939449270454666
Epoch -  26 Loss -  0.693935990080

{'W1': array([[0.09991715, 0.09991715],
        [0.09905522, 0.09905522]]),
 'b1': array([[-0.00025858],
        [-0.00025858]]),
 'W2': array([[0.09624859],
        [0.09624859]]),
 'b2': array([[-0.00775355]])}

### e. Keras baseline

In [34]:
model = Sequential([
    Input(shape=(2,)),
    Dense(2, activation='sigmoid'),
    Dense(1, activation='sigmoid')
])

In [35]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
new_weights = [np.array([[0.1, 0.1], [0.1, 0.1]], dtype=np.float32), np.array([0, 0], dtype=np.float32), np.array([[0.1], [0.1]], dtype=np.float32), np.array([0], dtype=np.float32)]

In [37]:
model.get_weights()

[array([[ 0.79536164,  0.2185775 ],
        [-0.77852213,  0.66463757]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[1.3994952],
        [0.8340105]], dtype=float32),
 array([0.], dtype=float32)]

In [38]:
model.set_weights(new_weights)
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [39]:
optimizer = keras.optimizers.SGD(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [40]:
model.fit(df.iloc[:,0:-1].values, df['placed'].values, epochs=50, verbose=1, batch_size=1)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6717  
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6828 
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7068 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7170 
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6563 
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6737 
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6829 
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7066 
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6840 
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6720 
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7167 
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7296 
Epoch 13/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6739 
Epoch 14/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7025 
Epoch 15/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7025 
Epoch 16/50
4/4 ━━━━━━━━━━━━━━━━━

In [41]:
model.get_weights()

[array([[0.09992012, 0.09992012],
        [0.09906398, 0.09906398]], dtype=float32),
 array([-0.0002576, -0.0002576], dtype=float32),
 array([[0.0962919],
        [0.0962919]], dtype=float32),
 array([-0.00770093], dtype=float32)]